# فصل ۷: ساخت برنامه‌های چت  
## شروع سریع API اوپن‌ای‌آی  

این دفترچه یادداشت از مخزن [نمونه‌های Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) اقتباس شده است که شامل دفترچه‌هایی است که به خدمات [Azure OpenAI](notebook-azure-openai.ipynb) دسترسی دارند.  

API پایتون اوپن‌ای‌آی همچنین با مدل‌های Azure OpenAI کار می‌کند، البته با چند تغییر کوچک. برای آشنایی بیشتر با تفاوت‌ها اینجا بخوانید: [چگونه بین نقاط پایانی OpenAI و Azure OpenAI با پایتون جابجا شویم](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)  


# مرور کلی  
«مدل‌های زبان بزرگ توابعی هستند که متن را به متن نگاشت می‌کنند. با داشتن یک رشته ورودی از متن، یک مدل زبان بزرگ تلاش می‌کند متنی که قرار است بعداً بیاید را پیش‌بینی کند»(1). این دفترچه یادداشت «شروع سریع» به کاربران مفاهیم سطح بالا در مورد LLM، نیازهای اصلی بسته برای شروع کار با AML، معرفی نرم‌افزاری به طراحی فرامین و چندین مثال کوتاه از کاربردهای مختلف را معرفی می‌کند.  


## فهرست مطالب  

[بررسی کلی](#overview)  
[چگونه از سرویس OpenAI استفاده کنیم](#how-to-use-openai-service)  
[1. ایجاد سرویس OpenAI خود](#1.-creating-your-openai-service)  
[2. نصب](#2.-installation)    
[3. مدارک](#3.-credentials)  

[موارد استفاده](#use-cases)    
[1. خلاصه کردن متن](#1.-summarize-text)  
[2. طبقه‌بندی متن](#2.-classify-text)  
[3. تولید نام‌های جدید محصول](#3.-generate-new-product-names)  
[4. تنظیم دقیق یک طبقه‌بند](#4.fine-tune-a-classifier)  

[مراجع](#references)


### اولین پرامپت خود را بسازید  
این تمرین کوتاه مقدمه‌ای پایه برای ارسال پرامپت‌ها به یک مدل OpenAI برای یک کار ساده «خلاصه‌سازی» ارائه می‌دهد.


**مراحل**:  
1. کتابخانه OpenAI را در محیط پایتون خود نصب کنید  
2. کتابخانه‌های کمکی استاندارد را بارگذاری کرده و اطلاعات امنیتی معمول OpenAI که برای سرویس OpenAI ساخته‌اید را تنظیم کنید  
3. مدلی را برای کار خود انتخاب کنید  
4. یک پرامپت ساده برای مدل بسازید  
5. درخواست خود را به API مدل ارسال کنید!


### ۱. نصب OpenAI


In [ ]:
%pip install openai python-dotenv

### ۲. وارد کردن کتابخانه‌های کمکی و نمونه‌سازی گواهی‌ها


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### ۳. یافتن مدل مناسب  
مدل‌هایی مانند GPT-4o و GPT-4o مینی می‌توانند زبان طبیعی را درک و تولید کنند و در پلتفرم OpenAI با سطوح مختلف قدرت و سرعت متناسب با وظایف گوناگون در دسترس هستند.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. طراحی پرامپت  

«جادوی مدل‌های زبان بزرگ این است که با آموزش دیده شدن برای کمینه کردن این خطای پیش‌بینی بر روی حجم زیادی از متن، مدل‌ها در نهایت مفاهیمی مفید برای این پیش‌بینی‌ها را یاد می‌گیرند. برای مثال، آن‌ها مفاهیمی مانند»(1):

* چگونه هجی کردن
* چگونه دستور زبان کار می‌کند
* چگونه پارافرایز کردن
* چگونه پاسخ به سوال‌ها دادن
* چگونه مکالمه کردن
* چگونه نوشتن به زبان‌های مختلف
* چگونه کد نویسی کردن
* و غیره.

#### چگونه یک مدل زبان بزرگ را کنترل کنیم  
«از میان تمام ورودی‌ها به یک مدل زبان بزرگ، به مراتب مؤثرترین آن پرامپت متنی است»(1).

مدل‌های زبان بزرگ را می‌توان به چند روش مختلف برای تولید خروجی هدایت کرد:

دستورالعمل: به مدل بگویید چه می‌خواهید
تکمیل: مدل را وادار کنید تا ابتدای آنچه می‌خواهید را تکمیل کند
نمایش: به مدل نشان دهید چه می‌خواهید، با یکی از این روش‌ها:
چند نمونه در داخل پرامپت
صدها یا هزاران نمونه در یک مجموعه داده آموزش تنظیم دقیق»



#### سه دستورالعمل پایه‌ای برای ایجاد پرامپت وجود دارد:

**نمایش دهید و توضیح دهید**. واضح کنید که چه می‌خواهید، چه از طریق دستورالعمل‌ها، چه نمونه‌ها، یا ترکیبی از هر دو. اگر می‌خواهید مدل یک فهرست از آیتم‌ها را به ترتیب حروف الفبا مرتب کند یا یک پاراگراف را بر اساس احساس طبقه‌بندی کند، به مدل نشان دهید که این همان چیزی است که می‌خواهید.

**داده‌های باکیفیت ارائه دهید**. اگر در حال ساخت یک طبقه‌بند هستید یا می‌خواهید مدل الگویی را دنبال کند، مطمئن شوید نمونه‌های کافی وجود دارد. حتماً نمونه‌های خود را بازبینی کنید — مدل معمولاً آن‌قدر هوشمند است که اشتباهات املایی ساده را می‌فهمد و پاسخ می‌دهد، اما ممکن است فرض کند که این کار عمدی است و این می‌تواند پاسخ را تحت تأثیر قرار دهد.

**تنظیمات خود را چک کنید.** تنظیمات temperature و top_p کنترل می‌کنند که مدل چقدر در تولید پاسخ قاطع و قطعی باشد. اگر پاسخ از مدل می‌خواهید که فقط یک جواب درست دارد، بهتر است این مقادیر پایین تنظیم شوند. اگر به دنبال پاسخ‌های متنوع‌تر هستید، می‌توانید آن‌ها را بالا ببرید. بزرگ‌ترین اشتباه مردم در این تنظیمات این است که فکر می‌کنند این‌ها کنترل‌های «باهوشی» یا «خلاقیت» مدل هستند.


منبع: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### ۵. ارسال کنید!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### تکرار همان تماس، نتایج چگونه مقایسه می‌شوند؟


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## خلاصه متن  
#### چالش  
متن را با افزودن «tl;dr:» به انتهای یک بخش متنی خلاصه کنید. دقت کنید که مدل چگونه می‌فهمد چگونه بدون دستورالعمل‌های اضافی چندین کار را انجام دهد. می‌توانید با استفاده از پیشنهادات توصیفی‌تر از «tl;dr» رفتار مدل را تغییر داده و خلاصه‌ای که دریافت می‌کنید را سفارشی کنید(3).  

کارهای اخیر با پیش‌ آموزش روی یک متن بزرگ و سپس تنظیم دقیق روی یک کار خاص، پیشرفت‌های قابل توجهی در بسیاری از وظایف و معیارهای NLP نشان داده‌اند. در حالی که معماری معمولاً بی‌تفاوت نسبت به کار است، این روش هنوز به مجموعه داده‌های تنظیم دقیق خاص کار با هزاران یا ده‌ها هزار نمونه نیاز دارد. در مقابل، انسان‌ها عموماً می‌توانند یک وظیفه زبانی جدید را فقط با چند مثال یا دستورالعمل‌های ساده انجام دهند - چیزی که سیستم‌های فعلی NLP هنوز تا حد زیادی در انجام آن مشکل دارند. در اینجا نشان می‌دهیم که افزایش مقیاس مدل‌های زبانی عملکرد چند-شات بی‌تفاوت نسبت به کار را به طور قابل توجهی بهبود می‌بخشد و گاهی حتی با روش‌های تنظیم دقیق پیشرفته رقابت می‌کند.  



خلاصه  


# تمرین‌ها برای چند مورد استفاده  
1. خلاصه‌سازی متن  
2. دسته‌بندی متن  
3. تولید نام‌های جدید محصول  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## دسته‌بندی متن  
#### چالش  
اقلام را در دسته‌هایی که در زمان استنتاج ارائه می‌شوند، دسته‌بندی کنید. در مثال زیر، ما هم دسته‌ها و هم متنی که باید دسته‌بندی شود را در درخواست ارائه می‌دهیم(*playground_reference). 

پرسش مشتری: سلام، یکی از کلیدهای صفحه‌کلید لپ‌تاپ من اخیراً شکسته است و من به یک جایگزین نیاز دارم:

دسته‌بندی شده:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## تولید نام‌های جدید محصول  
#### چالش  
ایجاد نام محصولات از کلمات نمونه. در اینجا اطلاعاتی درباره محصولی که قرار است برای آن نام تولید کنیم، در پرامپت وارد کرده‌ایم. همچنین یک مثال مشابه ارائه داده‌ایم تا الگوی مورد نظرمان را نشان دهیم. میزان دما را نیز بالا تنظیم کرده‌ایم تا تصادفی بودن و پاسخ‌های نوآورانه بیشتر شود.  

توضیح محصول: دستگاه ساخت میلک‌شیک خانگی  
کلمات پایه: سریع، سالم، جمع‌وجور.  
نام‌های محصول: HomeShaker, Fit Shaker, QuickShake, Shake Maker  

توضیح محصول: یک جفت کفش که می‌تواند به هر سایز پا بخورد.  
کلمات پایه: تطبیق‌پذیر، مناسب، همه‌سایزه.  


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# منابع  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [پورتال Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [بهترین روش‌ها برای تنظیم دقیق مدل‌های GPT برای طبقه‌بندی متن](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# برای کمک بیشتر  
[تیم تجاری‌سازی OpenAI](AzureOpenAITeam@microsoft.com) 


# مشارکت‌کنندگان
* لوئیس لی  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
